<a href="https://colab.research.google.com/github/blkout-hd/SWMCP/blob/main/mgk_sft_with_trl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Supervised Fine-tuning (SFT) with TRL

Fine-tuning requires a GPU. If you don't have one locally, you can run this notebook for free on [Google Colab](https://colab.research.google.com/github/Liquid4All/cookbook/blob/main/finetuning/notebooks/sft_with_trl.ipynb) using a free NVIDIA T4 GPU instance.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Liquid4All/cookbook/blob/main/finetuning/notebooks/sft_with_trl.ipynb)

### What's in this notebook?

In this notebook you will learn how to perform Supervised Fine-tuning (SFT) using the Hugging Face TRL (Transformer Reinforcement Learning) library.
We will use the [LFM2.5-1.2B-Instruct](https://docs.liquid.ai/docs/models/lfm25-1.2b-instruct) model and fine-tune it on the SmolTalk dataset. You'll learn both standard SFT and parameter-efficient fine-tuning with LoRA for constrained hardware.

We will cover
- Environment setup
- Data preparation
- Model training with TRL's SFTTrainer
- LoRA for parameter-efficient fine-tuning
- Local inference with your new model
- Model saving and exporting it into the format you need for **deployment**.

### Deployment options

LFM2.5 models are small and efficient, enabling deployment across a wide range of platforms:

<table align="left">
  <tr>
    <th>Deployment Target</th>
    <th>Use Case</th>
  </tr>
  <tr>
    <td>📱 <a href="https://docs.liquid.ai/leap/edge-sdk/android/android-quick-start-guide"><b>Android</b></a></td>
    <td>Mobile apps on Android devices</td>
  </tr>
  <tr>
    <td>📱 <a href="https://docs.liquid.ai/leap/edge-sdk/ios/ios-quick-start-guide"><b>iOS</b></a></td>
    <td>Mobile apps on iPhone/iPad</td>
  </tr>
  <tr>
    <td>🍎 <a href="https://docs.liquid.ai/docs/inference/mlx"><b>Apple Silicon Mac</b></a></td>
    <td>Local inference on Mac with MLX</td>
  </tr>
  <tr>
    <td>🦙 <a href="https://docs.liquid.ai/docs/inference/llama-cpp"><b>llama.cpp</b></a></td>
    <td>Local deployments on any hardware</td>
  </tr>
  <tr>
    <td>🦙 <a href="https://docs.liquid.ai/docs/inference/ollama"><b>Ollama</b></a></td>
    <td>Local inference with easy setup</td>
  </tr>
  <tr>
    <td>🖥️ <a href="https://docs.liquid.ai/docs/inference/lm-studio"><b>LM Studio</b></a></td>
    <td>Desktop app for local inference</td>
  </tr>
  <tr>
    <td>⚡ <a href="https://docs.liquid.ai/docs/inference/vllm"><b>vLLM</b></a></td>
    <td>Cloud deployments with high throughput</td>
  </tr>
  <tr>
    <td>☁️ <a href="https://docs.liquid.ai/docs/inference/modal-deployment"><b>Modal</b></a></td>
    <td>Serverless cloud deployment</td>
  </tr>
  <tr>
    <td>🏗️ <a href="https://docs.liquid.ai/docs/inference/baseten-deployment"><b>Baseten</b></a></td>
    <td>Production ML infrastructure</td>
  </tr>
  <tr>
    <td>🚀 <a href="https://docs.liquid.ai/docs/inference/fal-deployment"><b>Fal</b></a></td>
    <td>Fast inference API</td>
  </tr>
</table>

### Need help building with our models and tools?
Join the Liquid AI Discord Community and ask.

<a href="https://discord.com/invite/liquid-ai"><img src="https://img.shields.io/discord/1385439864920739850?color=7289da&label=Join%20Discord&logo=discord&logoColor=white" alt="Join Discord"></a>

And now, let the fine tune begin!

# 📦 Installation & Setup

First, let's install all the required packages:


In [ ]:
!pip install transformers==4.54.0 trl>=0.18.2 peft>=0.15.2

Let's now verify the packages are installed correctly

In [ ]:
import torch
import transformers
import trl
import os
os.environ["WANDB_DISABLED"] = "true"

print(f"📦 PyTorch version: {torch.__version__}")
print(f"🤗 Transformers version: {transformers.__version__}")
print(f"📊 TRL version: {trl.__version__}")

# Loading the model from Transformers 🤗



In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from IPython.display import display, HTML, Markdown
import torch

# Select a model to fine-tune from the list
lfm_models = [
    "LiquidAI/LFM2.5-1.2B-Instruct",
    "LiquidAI/LFM2.5-1.2B-JP",
    "LiquidAI/LFM2-8B-A1B",
    "LiquidAI/LFM2-2.6B-Exp",
    "LiquidAI/LFM2-2.6B",
    "LiquidAI/LFM2-700M",
    "LiquidAI/LFM2-350M",
]

# Model to fine-tune
model_id = "LiquidAI/LFM2.5-1.2B-Instruct"

print("📚 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("🧠 Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype="bfloat16",
#   attn_implementation="flash_attention_2" <- uncomment on compatible GPU
)

print("✅ Local model loaded successfully!")
print(f"🔢 Parameters: {model.num_parameters():,}")
print(f"📖 Vocab size: {len(tokenizer)}")
print(f"💾 Model size: ~{model.num_parameters() * 2 / 1e9:.1f} GB (bfloat16)")

# 🎯 Part 1: Supervised Fine-Tuning (SFT)

SFT teaches the model to follow instructions by training on input-output pairs (instruction vs response). This is the foundation for creating instruction-following models.

## Load an SFT Dataset

We will use [HuggingFaceTB/smoltalk](https://huggingface.co/datasets/HuggingFaceTB/smoltalk), limiting ourselves to the first 5k samples for brevity. Feel free to change the limit by changing the slicing index in the parameter `split`.

In [ ]:
from datasets import load_dataset

print("📥 Loading SFT dataset...")
train_dataset_sft = load_dataset("HuggingFaceTB/smoltalk", "all", split="train[:5000]")
eval_dataset_sft = load_dataset("HuggingFaceTB/smoltalk", "all", split="test[:500]")

print("✅ SFT Dataset loaded:")
print(f"   📚 Train samples: {len(train_dataset_sft)}")
print(f"   🧪 Eval samples: {len(eval_dataset_sft)}")
print(f"\n📝 Single Sample: {train_dataset_sft[0]['messages']}")

In [ ]:
# ========================================================================
# MCP-AUGMENTED DATASET LOADING & PREPROCESSING
# ========================================================================

from datasets import concatenate_datasets, Dataset
import json

print("📥 Loading MCP-focused reasoning datasets...")

# Load Qwen3.5 reasoning dataset (700x samples)
qwen_reasoning_ds = load_dataset("Jackrong/Qwen3.5-reasoning-700x", split="train")

# Load Claude Opus reasoning dataset (3000x filtered samples)
opus_reasoning_ds = load_dataset("nohurry/Opus-4.6-Reasoning-3000x-filtered", split="train")

print(f"✅ Loaded {len(qwen_reasoning_ds)} Qwen3.5 reasoning samples")
print(f"✅ Loaded {len(opus_reasoning_ds)} Opus reasoning samples")

In [ ]:
# ========================================================================
# MCP TOOL-CALLING AUGMENTATION WITH SEMANTIC INFERENCE
# ========================================================================

def create_mcp_augmented_samples(example):
    """
    Augment training samples with MCP tool-calling patterns emphasizing:
    - search_web (semantic web search)
    - find (element location with semantic inference)
    - computer (recursive action planning)
    - Tool chaining and recursive thinking
    """

    if "conversation" in example and len(example["conversation"]) >= 2:
        messages = example["conversation"]
        if messages[1]["from"] in ["gpt", "assistant"]:
            original_response = messages[1]["value"]

            # Add MCP-enhanced reasoning structure
            mcp_enhanced_response = f"""Let me approach this with MCP tool integration and recursive thinking:

**1. Semantic Analysis & Decomposition**
{original_response[:300] if len(original_response) > 300 else original_response}...

**2. Tool Selection & Strategy**
- Identify required MCP capabilities
- Map problem to tool primitives
- Plan recursive refinement approach

**3. Implementation**
{original_response}

**4. Validation & Recursive Improvement**
- Verify semantic correctness
- Apply inference for ambiguity resolution
- Iterate until convergence
"""

            messages[1]["value"] = mcp_enhanced_response

    return example

print("🔧 Augmenting datasets with MCP patterns...")
qwen_mcp = qwen_reasoning_ds.map(create_mcp_augmented_samples)
opus_mcp = opus_reasoning_ds.map(create_mcp_augmented_samples)
print("✅ MCP augmentation complete")

In [ ]:
# ========================================================================
# CREATE MCP-SPECIFIC TRAINING SAMPLES & COMBINE DATASETS
# ========================================================================

mcp_specific_samples = [
    {
        "conversation": [
            {"from": "human", "value": "Design an MCP server using search_web and find tools to perform semantic code search with recursive refinement."},
            {"from": "assistant", "value": """I'll design an MCP server with semantic search and recursive thinking:

**Semantic Decomposition:**
1. Parse user query into semantic components
2. Build query embedding space
3. Identify relevant tool primitives

**Tool Chain Design:**
<tool_call>
{"tool": "search_web", "parameters": {"queries": ["semantic code search patterns", "MCP tool implementation"]}}
</tool_call>

**Recursive Refinement Loop:**
```python
def semantic_search_recursive(query, depth=0, max_depth=5):
    if depth >= max_depth:
        return results

    # Search with semantic expansion
    candidates = mcp_search(query)

    # Analyze & refine
    refined_query = semantic_inference(query, candidates)

    # Recursive step
    return semantic_search_recursive(refined_query, depth+1)
```

**Implementation:** The MCP server maintains context across iterations, learning optimal search patterns through recursive feedback loops."""}
        ]
    },
    {
        "conversation": [
            {"from": "human", "value": "Implement tool chaining in MCP with semantic inference to automate complex web automation tasks."},
            {"from": "assistant", "value": """Let me design an MCP tool chain with semantic inference:

**1. Tool Dependency Analysis:**
- Identify required tool sequence
- Build semantic dependency graph
- Plan execution order

**2. Implementation with Recursive Planning:**
<tool_call>
{"tool": "computer", "parameters": {"actions": [{"action": "screenshot"}, {"action": "find", "query": "semantic target"}]}}
</tool_call>

**3. Semantic Inference Layer:**
- Interpret ambiguous instructions
- Infer missing parameters
- Validate tool outputs semantically

**4. Recursive Error Correction:**
When tools fail, the system:
1. Analyzes failure semantically
2. Identifies alternative approaches
3. Recursively retries with refinements

This creates robust, self-correcting automation."""}
        ]
    }
]

mcp_dataset = Dataset.from_list(mcp_specific_samples)
print(f"🧪 Created {len(mcp_dataset)} MCP-specific training samples")

# Combine all datasets
combined_dataset = concatenate_datasets([qwen_mcp, opus_mcp, mcp_dataset])
print(f"📊 Total combined training samples: {len(combined_dataset)}")

# Format for training
def format_for_training(example):
    if "conversation" in example:
        formatted = tokenizer.apply_chat_template(
            example["conversation"],
            tokenize=False,
            add_generation_prompt=False
        )
        return {"text": formatted}
    return example

train_dataset_formatted = combined_dataset.map(format_for_training)
print("✅ Dataset formatting complete!")

## Launch Training

We are now ready to launch an SFT run with `SFTTrainer`, feel free to modify `SFTConfig` to play around with different configurations.



In [ ]:
# ========================================================================
# MCP-OPTIMIZED SFT TRAINING CONFIGURATION
# ========================================================================

from trl import DataCollatorForCompletionOnlyLM

sft_config = SFTConfig(
    output_dir="./qwen3.5-mcp-reasoning",

    # Training dynamics optimized for reasoning
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,  # Effective batch size = 8

    # Learning rate with warmup for stable convergence
    learning_rate=2e-5,  # Lower LR for reasoning stability
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,

    # Optimization
    optim="adamw_torch_fused",
    weight_decay=0.01,
    max_grad_norm=1.0,

    # Logging & evaluation
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,

    # Memory optimization
    bf16=True,  # Use bfloat16 for numerical stability
    gradient_checkpointing=True,

    # Dataset specific
    max_seq_length=4096,  # Longer context for reasoning chains
    packing=False,  # Preserve reasoning structure

    dataset_text_field="text",
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

print("✅ MCP-optimized training configuration created")

# Response-only training (critical for reasoning quality)
response_template = "<|im_start|>assistant\n"
collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template,
    tokenizer=tokenizer
)

# Initialize trainer with MCP dataset
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset_formatted,
    eval_dataset=train_dataset_formatted.select(range(min(100, len(train_dataset_formatted)))),
    processing_class=tokenizer,
    data_collator=collator,
)

print("🏭 Trainer initialized with MCP-focused dataset")
print(f"📊 Training on {len(train_dataset_formatted)} samples")
print("🚀 Starting MCP-augmented training...")

# Train
training_output = trainer.train()

print("🎉 Training completed!")
print(f"📊 Final training loss: {training_output.training_loss:.4f}")

# Save model
trainer.save_model()
tokenizer.save_pretrained(sft_config.output_dir)
print(f"💾 Model saved to: {sft_config.output_dir}")

# 🎛️ Part 2: LoRA + SFT (Parameter-Efficient Fine-tuning)

LoRA (Low-Rank Adaptation) allows efficient fine-tuning by only training a small number of additional parameters. Perfect for limited compute resources!


## Wrap the model with PEFT

We specify target modules that will be finetuned while the rest of the models weights remains frozen. Feel free to modify the `r` (rank) value:
- higher -> better approximation of full-finetuning
- lower -> needs even less compute resources

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

GLU_MODULES = ["w1", "w2", "w3"]
MHA_MODULES = ["q_proj", "k_proj", "v_proj", "out_proj"]
CONV_MODULES = ["in_proj", "out_proj"]

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=8,  # <- lower values = fewer parameters
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=GLU_MODULES + MHA_MODULES + CONV_MODULES,
    bias="none",
    modules_to_save=None,
)

lora_model = get_peft_model(model, lora_config)
lora_model.print_trainable_parameters()

print("✅ LoRA configuration applied!")
print(f"🎛️  LoRA rank: {lora_config.r}")
print(f"📊 LoRA alpha: {lora_config.lora_alpha}")
print(f"🎯 Target modules: {lora_config.target_modules}")

## Launch Training

Now ready to launch the SFT training, but this time with the LoRA-wrapped model

In [ ]:
from trl import SFTConfig, SFTTrainer

lora_sft_config = SFTConfig(
    output_dir="./lfm2-sft-lora",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    learning_rate=5e-5,
    lr_scheduler_type="linear",
    warmup_steps=100,
    warmup_ratio=0.2,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    load_best_model_at_end=True,
    report_to=None,
)

print("🏗️  Creating LoRA SFT trainer...")
lora_sft_trainer = SFTTrainer(
    model=lora_model,
    args=lora_sft_config,
    train_dataset=train_dataset_sft,
    eval_dataset=eval_dataset_sft,
    processing_class=tokenizer,
)

print("\n🚀 Starting LoRA + SFT training...")
lora_sft_trainer.train()

print("🎉 LoRA + SFT training completed!")

lora_sft_trainer.save_model()
print(f"💾 LoRA model saved to: {lora_sft_config.output_dir}")

## Save merged model

Merge the extra weights learned with LoRA back into the model to obtain a "normal" model checkpoint.

In [ ]:
print("\n🔄 Merging LoRA weights...")
merged_model = lora_model.merge_and_unload()
merged_model.save_pretrained("./lfm2-lora-merged")
tokenizer.save_pretrained("./lfm2-lora-merged")
print("💾 Merged model saved to: ./lfm2-lora-merged")